# LC 743 — Network Delay Time
**Difficulty:** Medium | **Category:** Graph | **Pattern:** Dijkstra's Algorithm

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Run Dijkstra from source node k.
The answer is the maximum shortest-path distance across all
n nodes. If any node is unreachable, return -1.
</div>

## Official Problem Statement

You are given a network of `n` nodes, labeled from `1` to `n`.
You are also given `times`, a list of travel times as directed
edges `times[i] = (ui, vi, wi)`, where `ui` is the source node,
`vi` is the target node, and `wi` is the time it takes for a
signal to travel from source to target.

We will send a signal from a given node `k`. Return the
**minimum time** it takes for all the `n` nodes to receive the
signal. If it is impossible for all the `n` nodes to receive
the signal, return `-1`.

**Constraints:**
- `1 <= k <= n <= 100`
- `1 <= times.length <= 6000`
- `times[i].length == 3`
- `1 <= ui, vi <= n`
- `ui != vi`
- `0 <= wi <= 100`
- All `(ui, vi)` pairs are unique

## What This Is Actually Asking

Imagine you send a broadcast signal from one computer in a
network. Each link has a different delay.

The signal travels all paths simultaneously. Each node lights
up the moment the fastest path reaches it.

You want to know: when does the LAST node finally receive
the signal? That is the bottleneck — the slowest fastest-path.

If any node has no path from k, return -1 because the signal
can never reach everyone.

## Walk Through an Example by Hand

```
times = [[2,1,1],[2,3,1],[3,4,1]], n=4, k=2
```

Graph edges (directed, weighted):
```
2 --1--> 1
2 --1--> 3
3 --1--> 4
```

**Dijkstra steps from node 2:**

| Step | Pop      | dist array (1-indexed)    | Heap after pop           |
|------|----------|---------------------------|--------------------------|
| Init | —        | [inf, inf, 0, inf, inf]   | [(0,2)]                  |
| 1    | (0, 2)   | [inf, 1, 0, 1, inf]       | [(1,1),(1,3)]            |
| 2    | (1, 1)   | [inf, 1, 0, 1, inf]       | [(1,3)]  (node 1 done)   |
| 3    | (1, 3)   | [inf, 1, 0, 1, 2]         | [(2,4)]                  |
| 4    | (2, 4)   | [inf, 1, 0, 1, 2]         | []       (all done)      |

Max of dist[1..4] = max(1, 0, 1, 2) = **2** → answer is 2

## The Picture

**Weighted directed graph:**
```
         1
    +--------->
    |        Node 1
  Node 2
    |        Node 3 ----1----> Node 4
    +--------->
         1
```

More precisely:
```
  [2] --1--> [1]
   |
   1
   v
  [3] --1--> [4]
```

**Min-heap (cost, node) evolution:**
```
Start:   heap=[(0,2)]          dist=[inf,inf, 0,inf,inf]
Pop(2):  heap=[(1,1),(1,3)]    dist=[inf,  1, 0,  1,inf]
Pop(1):  heap=[(1,3)]          dist=[inf,  1, 0,  1,inf]
Pop(3):  heap=[(2,4)]          dist=[inf,  1, 0,  1,  2]
Pop(4):  heap=[]               dist=[inf,  1, 0,  1,  2]
                                           ^
                               Answer = max(1,0,1,2) = 2
```

Key: each pop finalizes the shortest path to that node.
We never revisit a settled node (skip if dist[node] < cost).

## When To Use This Pattern

- When you see **weighted directed graph + shortest path**,
  think Dijkstra with a min-heap.
- When you see **"all nodes reachable?"**, think: did every
  node get a finite distance? If not, return -1.
- When you see **"minimum time for broadcast to complete"**,
  think: max of all shortest-path distances.
- When weights are **non-negative**, Dijkstra is safe to use.
- When you need the **bottleneck** of a propagation, think
  max(shortest_paths).

## The Approach

Build an adjacency list from the times array. Initialize a
distance array of size n+1 to infinity, then set dist[k] = 0.

Push (0, k) onto a min-heap and run Dijkstra: pop the cheapest
node, skip it if already settled, then relax all its outgoing
edges by pushing updated (cost, neighbor) onto the heap.

After the heap empties, check dist[1..n]. If any value is
still infinity, return -1. Otherwise return the maximum value,
which is when the last node finally receives the signal.

In [ ]:
import heapq          # min-heap for Dijkstra priority queue
from typing import List  # type hints for function signatures

In [ ]:
def test_harness(func):
    """Run test cases for LC 743 - Network Delay Time."""
    tests = [
        # (times, n, k, expected)
        # Basic example from LeetCode
        ([[2,1,1],[2,3,1],[3,4,1]], 4, 2, 2),
        # Single node, source is itself
        ([[1,2,1]], 2, 1, 1),
        # Unreachable node — should return -1
        ([[1,2,1]], 2, 2, -1),
        # Single node network
        ([], 1, 1, 0),
        # Two paths to same node, take cheaper
        ([[1,2,10],[1,3,1],[3,2,1]], 3, 1, 2),
        # Linear chain
        ([[1,2,1],[2,3,2],[3,4,3]], 4, 1, 6),
    ]

    passed = 0
    for i, (times, n, k, expected) in enumerate(tests):
        result = func(times, n, k)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"got={result}, expected={expected}"
        )

    print(f"\nResult: {passed}/{len(tests)} tests passed")

In [ ]:
def network_delay_time(times: List[List[int]], n: int, k: int) -> int:
    """
    LC 743 - Network Delay Time.

    Restatement:
        Given directed weighted edges and a source node k,
        find the minimum time for all n nodes to receive a
        signal sent from k. Return -1 if any node unreachable.

    Approach:
        Dijkstra from k using a min-heap. Track shortest dist
        to each node. Skip already-settled nodes. After heap
        empties, return max(dist[1..n]) or -1 if any is inf.

    Time:  O((E + N) log N) — each edge/node processed once
    Space: O(N + E) — adjacency list + dist array + heap
    """
    pass


# Direct test prints — expected values shown in comments
print(network_delay_time([[2,1,1],[2,3,1],[3,4,1]], 4, 2))
# Expected: 2

print(network_delay_time([[1,2,1]], 2, 2))
# Expected: -1

print(network_delay_time([], 1, 1))
# Expected: 0

print(network_delay_time([[1,2,10],[1,3,1],[3,2,1]], 3, 1))
# Expected: 2

print(network_delay_time([[1,2,1],[2,3,2],[3,4,3]], 4, 1))
# Expected: 6

In [ ]:
# Uncomment and run when solution is ready
# test_harness(network_delay_time)

## Complexity

| Approach        | Time              | Space       |
|-----------------|-------------------|-------------|
| Brute Force BFS | O(N * (N + E))    | O(N + E)    |
| Dijkstra + Heap | O((E + N) log N)  | O(N + E)    |

## Real World Connection

At Citi, telemetry signals propagate across ~6,000 endpoints
in AWS VPC networks. When a health-check broadcast fires,
the system must confirm all nodes are reachable within an
SLA window — exactly the "max shortest path" problem here.

In AWS, VPC routing tables define directed weighted edges;
latency between availability zones is the edge weight.
Dijkstra's algorithm underlies how AWS Route 53 health checks
and traffic routing minimize end-to-end propagation delay.

A data engineer troubleshooting slow pipeline fan-out is
essentially asking: which downstream node takes the longest
to receive data from the source? That is LC 743 in production.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra